# 00. Project Overview — jp_llm_lab

## このプロジェクトは何か

日本語中心の**小型 Decoder-only Transformer** を PyTorch でゼロから実装し、
事前学習 → 計測 → 統制実験 → 較正 → SFT までを行う**教育用ラボ**です。

> **This project is not designed to build a competitive large language model.**
> 目的は高性能ではなく、言語モデルの内部機構・学習ダイナミクス・アーキテクチャの
> トレードオフ・確率較正・限界を**観測可能にし、理解すること**です。

設計原則は **Visualization-first**：主要な処理すべてに
「数式 → 小さな具体例 → tensor shape 追跡 → 可視化 → 実測 → 解釈 → 注意点」を付けます。

## 学習の地図（この教材で答えられるようになる問い）

| 問い | どこで扱うか |
|---|---|
| 言語モデルは何を学習しているか | NB04 (bigram) → NB13 (training dynamics) |
| 次token予測はどう行われるか | NB06 (forward pass), NB21 (generation anatomy) |
| tensorはどう変換されるか | NB06 (shape trace) |
| Attention/MLP/Residual/Normは何をするか | NB05, NB14, NB16 |
| 学習中にパラメータ・勾配・活性はどう変わるか | NB13, NB16, NB17 |
| モデルサイズ・構成の違いは何を変えるか | NB18 (ablation), NB19 (scaling) |
| 学習率・初期化・バッチサイズの影響 | NB09–NB12 |
| Loss/Perplexity/Calibrationの読み方 | NB13, NB20 |
| 事前学習とSFTの違い | NB23 |
| 「もっともらしい生成」と「本当の能力」の区別 | NB22 (memorization), NB19 |

**Milestone 1（本リリース）では NB00, 01, 03, 04, 05, 06 が完成済み**です。
NB02（コーパス探索）と NB07 以降は Milestone 2+ で追加されます。

## End-to-end データフロー（spec §8.1）

```text
Raw text 「私はその人を…」
  → Tokenizer（M1: 文字単位 / M2: BPE）
  → Token IDs                 [B, T]
  → Embeddings（token + position） [B, T, D]
  → Transformer blocks × N
      residual stream
        ├─ LayerNorm → Self-Attention → (+)
        └─ LayerNorm → MLP            → (+)
  → 最終 LayerNorm
  → LM head（token embeddingと重み共有）
  → Logits                    [B, T, V]
  → Softmax → 次token分布
  → Sampling（greedy / temperature / top-k / top-p）
  → 生成テキスト
```

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
# 環境の一言サマリ（詳細は NB01）
import jp_llm_lab
from jp_llm_lab.utils.env import collect_env_report
r = collect_env_report()
print(f"jp_llm_lab v{jp_llm_lab.__version__}")
print(f"GPU: {r.gpu_name} ({r.vram_gb} GB, BF16={r.bf16_supported})")
print(f"CUDA: {r.cuda_version} / SDPA backends: {r.sdpa_backends}")

jp_llm_lab v0.1.0
GPU: NVIDIA GeForce RTX 5080 (15.92 GB, BF16=True)
CUDA: 12.8 / SDPA backends: ['math', 'flash', 'mem_efficient', 'cudnn']


## リポジトリの歩き方

```text
jp_llm_lab/
├── src/jp_llm_lab/     # すべて手書きの実装（attention・学習ループ・計測フック…）
├── scripts/            # 実験の実行入口（configから再実行可能）
├── configs/            # 実験設定 YAML
├── experiments/runs/   # 実行成果物（metrics.jsonl / checkpoints / samples）
├── notebooks/          # 本教材（tools/build_notebooks.py から生成）
├── reports/html/       # 静的HTMLレポート（make report）
└── tests/              # 44 tests（因果マスク・SDPA一致・過学習 等）
```

### Milestone 1 の再現手順

```bash
make -C jp_llm_lab corpus       # 青空文庫サンプル取得
make -C jp_llm_lab test         # テスト
make -C jp_llm_lab train-bigram # bigram ベースライン
make -C jp_llm_lab train-smoke  # Model S (1.1M) 学習 (~40秒 on RTX 5080)
make -C jp_llm_lab bench        # explicit vs SDPA
make -C jp_llm_lab notebooks    # 本ノートブック群の再生成・実行
make -C jp_llm_lab report       # HTMLレポート
```

## 測定の信頼性ポリシー

- **実測と例示を混ぜない**: 全run は `runmeta.json`（git hash・パッケージ版・GPU・時刻）と
  `config.json`（seed・ハイパーパラメータ）を保存。コーパスは `data/manifests/` に
  URL・ライセンス・sha256 を記録し、合成フォールバック使用時は `synthetic: true` を明記。
- **再学習なしで分析可能**: ノートブックとレポートは保存済み成果物のみを読む。
- **仮説に合わない結果もそのまま**: 例として、レポートの attention 分析では
  「文頭への注意が増える」という事前予想が**成り立たなかった**ことを数値ごと記載している。

## 確認問題

1. このプロジェクトの「完了」は何で定義されるか（性能か、観測可能性か）。
2. `experiments/runs/` に保存される3種類のレコード（train/eval/checkpoint）の役割は。
3. なぜ Milestone 1 では 30M モデルではなく 1M モデルから始めるのか。

**次へ**: [01_environment_and_gpu](01_environment_and_gpu.ipynb) — ハードウェアを実測し、設定を根拠づける。